# 03 — Feature Extraction with Google Earth Engine

This notebook extracts remote sensing features at labeled mining/non-mining point locations using Google Earth Engine. We build a multi-band feature stack from:

- **Sentinel-2** (optical): 10 spectral bands + 3 spectral indices (NDVI, NDWI, BSI)
- **Sentinel-1** (SAR): VV, VH polarizations + VV/VH ratio
- **SRTM** (terrain): elevation, slope, aspect

The result is a tabular dataset with ~19 features per labeled point, saved as `features_merged.csv`.

## 1. Setup and Imports

In [50]:
import ee
import geemap
import pandas as pd
import geopandas as gpd
import numpy as np
import os
import math
import warnings

warnings.filterwarnings("ignore")

In [51]:
# Initialize Google Earth Engine
# If not authenticated, run: ee.Authenticate()
try:
    ee.Initialize(project='concise-ion-489212-a9')
    print("GEE initialized successfully.")
except Exception as e:
    print(f"Initialization failed: {e}")
    print("Attempting authentication...")
    ee.Authenticate()
    ee.Initialize(project='concise-ion-489212-a9')
    print("GEE initialized after authentication.")

GEE initialized successfully.


## 2. Load Labeled Points

In [52]:
# Load the labeled points GeoDataFrame
gdf = gpd.read_file("../data/processed/all_labeled_points.geojson")

print(f"Total labeled points: {len(gdf)}")
print(f"CRS: {gdf.crs}")
print(f"Columns: {list(gdf.columns)}")
print(f"\nLabel distribution:")
print(gdf["label"].value_counts())
gdf.head()

Total labeled points: 5704
CRS: EPSG:4326
Columns: ['label', 'geometry']

Label distribution:
label
1    2888
0    2816
Name: count, dtype: int64


,label,geometry
0,1,POINT (28.71526 0.33702)
1,1,POINT (28.69916 0.32153)
2,1,POINT (28.18514 0.54499)
3,1,POINT (28.88453 -0.35253)
4,1,POINT (28.90394 -0.03671)


In [53]:
# Compute bounding box of the study area for filtering satellite imagery
bounds = gdf.total_bounds  # [minx, miny, maxx, maxy]
# Add a small buffer (0.5 degrees) around the extent
buffer_deg = 0.5
study_area = ee.Geometry.Rectangle([
    bounds[0] - buffer_deg,
    bounds[1] - buffer_deg,
    bounds[2] + buffer_deg,
    bounds[3] + buffer_deg
])

print(f"Study area bounds (with buffer):")
print(f"  minx: {bounds[0] - buffer_deg:.4f}")
print(f"  miny: {bounds[1] - buffer_deg:.4f}")
print(f"  maxx: {bounds[2] + buffer_deg:.4f}")
print(f"  maxy: {bounds[3] + buffer_deg:.4f}")

Study area bounds (with buffer):
  minx: 24.8084
  miny: -5.4939
  maxx: 31.2343
  maxy: 3.4992


## 3. Sentinel-2 Composite (Optical)

We use the Sentinel-2 Level-2A Surface Reflectance (Harmonized) collection for 2023. Cloud masking is performed using the companion S2 Cloud Probability collection (`COPERNICUS/S2_CLOUD_PROBABILITY`), which provides per-pixel cloud probability scores. Pixels with cloud probability above 50% are masked out before computing the annual median composite.

In [54]:
# Define date range
start_date = "2023-01-01"
end_date = "2023-12-31"

# Load Sentinel-2 SR Harmonized collection
# Pre-filter with CLOUDY_PIXEL_PERCENTAGE < 50 to reduce collection from ~10K to ~3K images
# This dramatically reduces the computation graph size
s2_sr = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate(start_date, end_date)
    .filterBounds(study_area)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 50))
)

# Load S2 Cloud Probability collection (also filtered to match)
s2_cloud_prob = (
    ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
    .filterDate(start_date, end_date)
    .filterBounds(study_area)
)

print(f"S2 SR collection size (pre-filtered <50% cloud): {s2_sr.size().getInfo()}")
print(f"S2 Cloud Probability collection size: {s2_cloud_prob.size().getInfo()}")

S2 SR collection size (pre-filtered <50% cloud): 4538
S2 Cloud Probability collection size: 11339


In [55]:
# Join the S2 SR collection with the Cloud Probability collection
# The join links each S2 image with its corresponding cloud probability image
# based on the shared 'system:index' property.

join_filter = ee.Filter.equals(
    leftField="system:index",
    rightField="system:index"
)

joined = ee.ImageCollection(
    ee.Join.saveFirst("cloud_prob").apply(
        primary=s2_sr,
        secondary=s2_cloud_prob,
        condition=join_filter
    )
)

print(f"Joined collection size: {joined.size().getInfo()}")

Joined collection size: 4492


In [56]:
def mask_s2_clouds(image):
    """Mask clouds using s2cloudless cloud probability.
    Pixels with cloud probability > 50 are masked out.
    """
    cloud_prob_img = ee.Image(image.get("cloud_prob"))
    is_not_cloud = cloud_prob_img.select("probability").lt(50)
    return image.updateMask(is_not_cloud)


# Apply cloud masking and compute the annual median composite
# NO .clip() — reduces computation graph; sampleRegions handles spatial filtering
s2_bands = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]

s2_composite = (
    joined
    .map(mask_s2_clouds)
    .select(s2_bands)
    .median()
)

print(f"S2 composite bands: {s2_composite.bandNames().getInfo()}")

S2 composite bands: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']


In [57]:
# Compute spectral indices

# NDVI = (B8 - B4) / (B8 + B4)
ndvi = s2_composite.normalizedDifference(["B8", "B4"]).rename("NDVI")

# NDWI = (B3 - B8) / (B3 + B8)
ndwi = s2_composite.normalizedDifference(["B3", "B8"]).rename("NDWI")

# BSI (Bare Soil Index) = ((B11 + B4) - (B8 + B2)) / ((B11 + B4) + (B8 + B2))
bsi = (
    s2_composite.select("B11").add(s2_composite.select("B4"))
    .subtract(s2_composite.select("B8").add(s2_composite.select("B2")))
    .divide(
        s2_composite.select("B11").add(s2_composite.select("B4"))
        .add(s2_composite.select("B8").add(s2_composite.select("B2")))
    )
    .rename("BSI")
)

# Add indices to the S2 composite
s2_composite = s2_composite.addBands([ndvi, ndwi, bsi])

print(f"S2 composite with indices — bands: {s2_composite.bandNames().getInfo()}")
print(f"Total S2 bands: {s2_composite.bandNames().size().getInfo()}")

S2 composite with indices — bands: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'NDVI', 'NDWI', 'BSI']
Total S2 bands: 13


## 4. Sentinel-1 SAR Composite

Sentinel-1 provides C-band Synthetic Aperture Radar (SAR) data that is weather- and daylight-independent. We use the Ground Range Detected (GRD) product in Interferometric Wide (IW) swath mode with ascending orbit direction. The VV and VH polarization bands are selected, and a VV/VH ratio band is computed to capture surface scattering properties.

In [58]:
# Load Sentinel-1 GRD collection
s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterDate(start_date, end_date)
    .filterBounds(study_area)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.eq("orbitProperties_pass", "ASCENDING"))
    .select(["VV", "VH"])
)

print(f"S1 collection size: {s1.size().getInfo()}")

# Compute annual median composite (no .clip())
s1_composite = s1.median()

# Compute VV/VH ratio
vv_vh_ratio = (
    s1_composite.select("VV")
    .subtract(s1_composite.select("VH"))
    .rename("VV_VH_ratio")
)

s1_composite = s1_composite.addBands(vv_vh_ratio)

print(f"S1 composite bands: {s1_composite.bandNames().getInfo()}")

S1 collection size: 365
S1 composite bands: ['VV', 'VH', 'VV_VH_ratio']


## 5. SRTM Terrain Features

The Shuttle Radar Topography Mission (SRTM) digital elevation model provides elevation data at ~30m resolution. We derive slope and aspect using `ee.Terrain.products()`. These terrain variables help distinguish mining sites, which are often located on specific topographic positions.

In [59]:
# Load SRTM elevation
srtm = ee.Image("USGS/SRTMGL1_003")

# Compute terrain products (slope, aspect, hillshade)
terrain = ee.Terrain.products(srtm)

# Select elevation, slope, and aspect (no .clip())
elevation = srtm.select("elevation")
slope = terrain.select("slope")
aspect = terrain.select("aspect")

srtm_composite = elevation.addBands([slope, aspect])

print(f"SRTM composite bands: {srtm_composite.bandNames().getInfo()}")

SRTM composite bands: ['elevation', 'slope', 'aspect']


## 6. Stack All Bands into a Single Multi-Band Image

We combine all features into a single image with ~19 bands:

| Source | Bands | Count |
|--------|-------|-------|
| Sentinel-2 | B2, B3, B4, B5, B6, B7, B8, B8A, B11, B12, NDVI, NDWI, BSI | 13 |
| Sentinel-1 | VV, VH, VV_VH_ratio | 3 |
| SRTM | elevation, slope, aspect | 3 |
| **Total** | | **19** |

In [60]:
# Stack all bands into one multi-band image
stacked_image = s2_composite.addBands(s1_composite).addBands(srtm_composite)

band_names = stacked_image.bandNames().getInfo()
print(f"Stacked image bands ({len(band_names)} total):")
for i, name in enumerate(band_names, 1):
    print(f"  {i:2d}. {name}")

Stacked image bands (19 total):
   1. B2
   2. B3
   3. B4
   4. B5
   5. B6
   6. B7
   7. B8
   8. B8A
   9. B11
  10. B12
  11. NDVI
  12. NDWI
  13. BSI
  14. VV
  15. VH
  16. VV_VH_ratio
  17. elevation
  18. slope
  19. aspect


## 7. Extract Features at Labeled Points

We split the extraction into **separate exports** to stay within GEE community tier memory limits:
1. Sentinel-2 features (13 bands) — split into **batches of 2000 points** to avoid memory errors
2. Sentinel-1 features (3 bands) — single export
3. SRTM features (3 bands) — single export (single image, no compositing)

Each export runs server-side on Google Drive. After all complete, we merge them locally.

In [61]:
def gdf_to_ee_fc(gdf):
    """Convert a GeoDataFrame of points to an ee.FeatureCollection."""
    features = []
    for idx, row in gdf.iterrows():
        geom = ee.Geometry.Point([row.geometry.x, row.geometry.y])
        props = {"point_id": int(idx), "label": int(row["label"])}
        features.append(ee.Feature(geom, props))
    return ee.FeatureCollection(features)

# Convert all labeled points
ee_points = gdf_to_ee_fc(gdf)
print(f"Converted {ee_points.size().getInfo()} points to ee.FeatureCollection")

Converted 5704 points to ee.FeatureCollection


In [67]:
# --- Export 1: Sentinel-2 (13 bands) — batched to avoid memory limits ---
# Change BATCH_ID to 0, 1, 2, ... and re-run this cell for each batch
BATCH_ID = 1
BATCH_SIZE = 1000

start_idx = BATCH_ID * BATCH_SIZE
end_idx = min((BATCH_ID + 1) * BATCH_SIZE, len(gdf))
batch_gdf = gdf.iloc[start_idx:end_idx]
batch_fc = gdf_to_ee_fc(batch_gdf)

sampled_s2 = s2_composite.sampleRegions(
    collection=batch_fc, scale=100, geometries=False
)

task_s2 = ee.batch.Export.table.toDrive(
    collection=sampled_s2,
    description=f'features_s2_batch{BATCH_ID}',
    fileNamePrefix=f'features_s2_batch{BATCH_ID}',
    fileFormat='CSV'
)
task_s2.start()

n_batches = math.ceil(len(gdf) / BATCH_SIZE)
print(f"S2 batch {BATCH_ID}/{n_batches-1}: points {start_idx}–{end_idx-1} ({end_idx - start_idx} points)")
print(f"Change BATCH_ID to {BATCH_ID + 1} and re-run for the next batch." if BATCH_ID < n_batches - 1 else "This is the last batch.")

S2 batch 1/5: points 1000–1999 (1000 points)
Change BATCH_ID to 2 and re-run for the next batch.


In [48]:
# --- Export 2: Sentinel-1 (3 bands) ---
sampled_s1 = s1_composite.sampleRegions(
    collection=ee_points, scale=100, geometries=False
)

task_s1 = ee.batch.Export.table.toDrive(
    collection=sampled_s1,
    description='features_s1',
    fileNamePrefix='features_s1',
    fileFormat='CSV'
)
task_s1.start()
print("Export 2/3 started: Sentinel-1 (3 bands) → features_s1.csv")

# --- Export 3: SRTM (3 bands) ---
sampled_srtm = srtm_composite.sampleRegions(
    collection=ee_points, scale=100, geometries=False
)

task_srtm = ee.batch.Export.table.toDrive(
    collection=sampled_srtm,
    description='features_srtm',
    fileNamePrefix='features_srtm',
    fileFormat='CSV'
)
task_srtm.start()
print("Export 3/3 started: SRTM (3 bands) → features_srtm.csv")

print("\nAll 3 exports running! Monitor at: https://code.earthengine.google.com/tasks")
print("When all complete, download all 3 CSVs from Google Drive → ../data/processed/")

Export 2/3 started: Sentinel-1 (3 bands) → features_s1.csv
Export 3/3 started: SRTM (3 bands) → features_srtm.csv

All 3 exports running! Monitor at: https://code.earthengine.google.com/tasks
When all complete, download all 3 CSVs from Google Drive → ../data/processed/


In [68]:
# Check export task status for current S2 batch + S1/SRTM
status = task_s2.status()
state = status['state']
msg = f"  S2 batch {BATCH_ID}: {state}"
if state == 'FAILED':
    msg += f" — {status.get('error_message', 'unknown error')}"
print(msg)

for name, task in [("S1", task_s1), ("SRTM", task_srtm)]:
    status = task.status()
    state = status['state']
    msg = f"  {name}: {state}"
    if state == 'FAILED':
        msg += f" — {status.get('error_message', 'unknown error')}"
    print(msg)

  S2 batch 1: COMPLETED
  S1: COMPLETED
  SRTM: COMPLETED


## 8. Merge Exported CSVs and Handle Null Values

Once all 3 Google Drive exports complete, download the CSVs and place them in `../data/processed/`:
- `features_s2.csv`
- `features_s1.csv`
- `features_srtm.csv`

We merge them on `point_id` and `label`, then clean null values.

In [69]:
# Load and merge the exported CSVs
# S2 was exported in batches — concatenate them
import glob

s2_files = sorted(glob.glob("../data/processed/features_s2_batch*.csv"))
if s2_files:
    df_s2 = pd.concat([pd.read_csv(f) for f in s2_files], ignore_index=True)
    print(f"S2: loaded {len(s2_files)} batch file(s)")
else:
    # Fallback to single file from previous runs
    df_s2 = pd.read_csv("../data/processed/features_s2.csv")
    print("S2: loaded single file (legacy)")

df_s1 = pd.read_csv("../data/processed/features_s1.csv")
df_srtm = pd.read_csv("../data/processed/features_srtm.csv")

print(f"S2:   {df_s2.shape} — columns: {list(df_s2.columns)}")
print(f"S1:   {df_s1.shape} — columns: {list(df_s1.columns)}")
print(f"SRTM: {df_srtm.shape} — columns: {list(df_srtm.columns)}")

# Merge on point_id and label
merge_keys = ["point_id", "label"]
features_df = df_s2.merge(df_s1, on=merge_keys, how="outer")
features_df = features_df.merge(df_srtm, on=merge_keys, how="outer")

# Drop any GEE system columns and .geo columns (including merge suffixes like .geo_x, .geo_y)
drop_cols = [c for c in features_df.columns if c.startswith("system:") or c.startswith(".geo")]
features_df = features_df.drop(columns=drop_cols, errors="ignore")

# Drop duplicate _1 columns created by addBands() in the S2 export
# (NDVI, NDWI, BSI were added twice — once in the composite, once explicitly)
dup_cols = [c for c in features_df.columns if c.endswith("_1") and c[:-2] in features_df.columns]
if dup_cols:
    print(f"\nDropping {len(dup_cols)} duplicate columns: {dup_cols}")
    features_df = features_df.drop(columns=dup_cols)

# Add real lat/lon coordinates from the original labeled points for spatial blocking
features_df["lat"] = gdf.geometry.y.values[features_df["point_id"].values]
features_df["lon"] = gdf.geometry.x.values[features_df["point_id"].values]

print(f"\nMerged: {features_df.shape}")
features_df.head()

S2: loaded 6 batch file(s)
S2:   (5704, 17) — columns: ['system:index', 'B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'BSI', 'NDVI', 'NDWI', 'label', 'point_id', '.geo']
S1:   (2437, 7) — columns: ['system:index', 'VH', 'VV', 'VV_VH_ratio', 'label', 'point_id', '.geo']
SRTM: (5704, 7) — columns: ['system:index', 'aspect', 'elevation', 'label', 'point_id', 'slope', '.geo']

Merged: (5704, 23)


,B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A,...,label,point_id,VH,VV,VV_VH_ratio,aspect,elevation,slope,lat,lon
0,1501.0,662.0,339.0,505.0,304.0,782.0,2382.0,2939.0,2937.0,3239.0,...,1,0,-13.856659,-7.145652,6.711007,0,983,2,0.337021,28.715262
1,1934.0,881.5,430.5,614.0,392.0,969.5,2689.5,3221.0,3298.0,3578.5,...,1,1,-12.660043,-6.595626,6.064417,150,1026,10,0.321530,28.699160
2,1323.0,582.0,328.0,416.0,278.0,680.0,1962.0,2492.0,2486.0,2753.0,...,1,2,-13.241170,-6.643131,6.598038,305,768,6,0.544992,28.185142
3,1549.0,755.0,328.0,519.0,307.0,836.0,2213.0,2637.0,2626.0,2881.0,...,1,3,-12.169551,-5.409893,6.759657,5,1330,6,-0.352529,28.884528
4,1943.0,893.5,346.0,535.5,331.5,932.5,2907.0,3596.5,3615.0,3941.5,...,1,4,-14.135198,-7.807693,6.327505,221,1171,4,-0.036707,28.903945


In [70]:
# Expected feature columns
expected_bands = [
    "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12",
    "NDVI", "NDWI", "BSI",
    "VV", "VH", "VV_VH_ratio",
    "elevation", "slope", "aspect"
]

feature_cols = [c for c in expected_bands if c in features_df.columns]
missing = [c for c in expected_bands if c not in features_df.columns]

print(f"Feature columns found: {len(feature_cols)}/{len(expected_bands)}")
if missing:
    print(f"WARNING — Missing: {missing}")

# Report null counts
null_counts = features_df[feature_cols].isnull().sum()
null_pct = (null_counts / len(features_df) * 100).round(1)

null_report = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
null_report = null_report[null_report["null_count"] > 0].sort_values("null_count", ascending=False)

if len(null_report) > 0:
    print(f"\nColumns with nulls:")
    print(null_report.to_string())
else:
    print("\nNo null values found!")

Feature columns found: 19/19

Columns with nulls:
             null_count  null_pct
VV                 3267      57.3
VH                 3267      57.3
VV_VH_ratio        3267      57.3


In [71]:
# Drop rows with more than 50% null features, fill remaining with median
n_before = len(features_df)
max_nulls = len(feature_cols) // 2
row_nulls = features_df[feature_cols].isnull().sum(axis=1)
features_df = features_df[row_nulls <= max_nulls].copy()
print(f"Dropped {n_before - len(features_df)} rows with >{max_nulls} null features.")

# Fill remaining nulls with column median
for col in feature_cols:
    if features_df[col].isnull().any():
        median_val = features_df[col].median()
        n_filled = features_df[col].isnull().sum()
        features_df[col] = features_df[col].fillna(median_val)
        print(f"  Filled {n_filled} nulls in '{col}' with median = {median_val:.4f}")

print(f"\nClean dataset: {len(features_df)} rows, {len(feature_cols)} features")

Dropped 0 rows with >9 null features.
  Filled 3267 nulls in 'VV' with median = -8.0165
  Filled 3267 nulls in 'VH' with median = -14.4808
  Filled 3267 nulls in 'VV_VH_ratio' with median = 6.4077

Clean dataset: 5704 rows, 19 features


## 9. Save Cleaned Features

Overwrite the raw export with the cleaned version (nulls handled) for use in notebook 04.

In [72]:
# Verify label column is present
assert "label" in features_df.columns, "ERROR: 'label' column missing from exported data!"

print(f"Label distribution:")
print(features_df["label"].value_counts())
print(f"\nFeature summary:")
features_df[feature_cols].describe().round(2)

Label distribution:
label
1    2888
0    2816
Name: count, dtype: int64

Feature summary:


,B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12,NDVI,NDWI,BSI,VV,VH,VV_VH_ratio,elevation,slope,aspect
count,5704.00,5704.00,5704.00,5704.00,5704.00,5704.00,5704.00,5704.00,5704.00,5704.00,5704.00,5704.00,5704.00,5704.00,5704.00,5704.00,5704.00,5704.00,5704.00
mean,408.92,604.20,506.37,973.46,2273.89,2766.55,2753.20,3016.27,1801.99,955.51,0.70,-0.64,-0.17,-8.05,-14.53,6.45,1015.77,7.82,176.65
std,150.75,223.04,335.89,351.43,386.15,452.25,451.92,491.35,509.44,449.82,0.17,0.12,0.13,1.65,1.45,0.47,494.93,6.80,104.42
min,148.00,219.00,170.00,173.00,112.00,107.00,76.00,72.00,105.00,85.00,-0.48,-0.82,-0.39,-24.76,-28.01,2.05,419.00,0.00,0.00
25%,308.50,444.00,283.00,709.00,2006.00,2477.00,2472.00,2719.88,1434.38,632.38,0.62,-0.72,-0.27,-8.02,-14.48,6.41,643.00,3.00,87.00
50%,366.00,535.50,358.00,860.00,2247.75,2739.00,2724.75,2999.50,1690.75,783.25,0.78,-0.69,-0.23,-8.02,-14.48,6.41,841.50,6.00,180.00
75%,461.50,708.00,627.00,1158.00,2518.62,3047.25,3032.12,3321.62,2082.62,1172.00,0.81,-0.59,-0.09,-8.02,-14.48,6.41,1274.00,10.00,270.00
max,2658.00,3357.00,4333.00,4883.00,4972.50,5083.00,4867.50,5036.00,4703.00,3799.00,0.89,0.59,0.29,6.69,-0.91,11.50,4127.00,48.00,359.00


In [73]:
# Save the cleaned version (overwrites the raw export)
output_path = "../data/processed/features_merged.csv"
features_df.to_csv(output_path, index=False)

print(f"Saved cleaned features to: {output_path}")
print(f"  Rows: {len(features_df)}")
print(f"  Columns: {len(features_df.columns)}")
print(f"  Feature columns: {len(feature_cols)}")
print(f"  File size: {os.path.getsize(output_path) / 1024:.1f} KB")

Saved cleaned features to: ../data/processed/features_merged.csv
  Rows: 5704
  Columns: 23
  Feature columns: 19
  File size: 1176.0 KB


In [74]:
# Final verification
verify_df = pd.read_csv(output_path)
print(f"Verified: {len(verify_df)} rows, {len(verify_df.columns)} columns.")
print(f"Ready for notebook 04 (model training).")

Verified: 5704 rows, 23 columns.
Ready for notebook 04 (model training).
